In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ------------------------------
# Dataset
# ------------------------------
class HandSignDataset(Dataset):
    def __init__(self, data_dir="data/raw"):
        self.samples = []
        self.labels = []
        self.class_counts = {}
        self.label_map = {chr(i+65): i for i in range(26)}  # A=0 ... Z=25

        # Make sure data directory exists
        os.makedirs(data_dir, exist_ok=True)

        for letter, idx in self.label_map.items():
            path = os.path.join(data_dir, f"{letter}.npy")
            if os.path.exists(path):
                data = np.load(path)
                if data.ndim != 2 or data.shape[1] != 63:
                    raise ValueError(f"Invalid samples in {path}: expected shape (samples, 63), got {data.shape}")
                if len(data) == 0:
                    continue
                self.samples.extend(data)
                self.labels.extend([idx] * len(data))
                self.class_counts[letter] = len(data)

        if len(self.samples) == 0:
            raise RuntimeError(f"No training data found in {data_dir}. "
                               f"Collect samples in collect_data.ipynb first!")

        self.samples = torch.tensor(np.array(self.samples), dtype=torch.float32)
        self.labels = torch.tensor(np.array(self.labels), dtype=torch.long)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]


# ------------------------------
# Model
# ------------------------------
class HandSignModel(nn.Module):
    def __init__(self, input_size=63, num_classes=26):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


# ------------------------------
# Training
# ------------------------------
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    dataset = HandSignDataset()
    print(f"Loaded {len(dataset)} fresh samples from {len(dataset.class_counts)} letters:")
    print(", ".join(f"{letter}={count}" for letter, count in sorted(dataset.class_counts.items())))
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    model = HandSignModel().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 25
    for epoch in range(epochs):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    # ✅ make sure models folder exists
    os.makedirs("models", exist_ok=True)
    torch.save(model.state_dict(), "models/best_model.pt")
    print("✅ Model saved at models/best_model.pt")


if __name__ == "__main__":
    main()


Using device: cpu
Loaded 556 fresh samples from 4 letters:
E=240, H=104, L=104, O=108
Epoch 1/25, Loss: 3.1290
Epoch 2/25, Loss: 2.5367
Epoch 3/25, Loss: 1.5540
Epoch 4/25, Loss: 1.2810
Epoch 5/25, Loss: 1.2101
Epoch 6/25, Loss: 1.1629
Epoch 7/25, Loss: 1.1038
Epoch 8/25, Loss: 1.0592
Epoch 9/25, Loss: 0.9737
Epoch 10/25, Loss: 0.9241
Epoch 11/25, Loss: 0.8720
Epoch 12/25, Loss: 0.7939
Epoch 13/25, Loss: 0.7198
Epoch 14/25, Loss: 0.6504
Epoch 15/25, Loss: 0.5699
Epoch 16/25, Loss: 0.5031
Epoch 17/25, Loss: 0.4347
Epoch 18/25, Loss: 0.3792
Epoch 19/25, Loss: 0.3283
Epoch 20/25, Loss: 0.2717
Epoch 21/25, Loss: 0.2238
Epoch 22/25, Loss: 0.1875
Epoch 23/25, Loss: 0.1582
Epoch 24/25, Loss: 0.1299
Epoch 25/25, Loss: 0.1077
✅ Model saved at models/best_model.pt
